# Estudio de detección de similitud entre programas Python
## Cuaderno 1. Curaduría, muestreo y verificación de datos

**Fuente:** PoolC, repositorio `PoolC/1-fold-clone-detection-600k-5fold`, publicado en Hugging Face.  
**Objeto de estudio:** clasificación binaria de pares de código Python.  
**Unidad de análisis:** dos fragmentos de código y una etiqueta de similitud.  
**Fecha del experimento:** 30 de octubre de 2026.

### Objetivo metodológico

Establecer un conjunto experimental trazable, balanceado y reproducible, evitando que fragmentos idénticos o grupos de procedencia compartidos aparezcan simultáneamente en entrenamiento y evaluación. Las etiquetas describen similitud dentro de PoolC, no prueban plagio académico.

El notebook reconoce dos escenarios. Si los archivos Parquet originales están disponibles, permite **reconstruir la muestra** mediante lectura por lotes y muestreo por reservorio. Si únicamente se dispone de `poolc_prepared.zip`, ejecuta una **auditoría reproducible del conjunto previamente exportado**, sin afirmar que volvió a extraer los registros originales. Ambos procedimientos conservan las mismas comprobaciones de calidad.

## 1. Entorno y trazabilidad de la fuente

Se emplean bibliotecas abiertas para leer Parquet por lotes, gestionar tablas y establecer identificadores criptográficos de los fragmentos. La fuente tiene siete variables originales: `code1`, `code2`, `similar`, `pair_id`, `question_pair_id`, `code1_group` y `code2_group`. Las variables de grupo e identificación se reservan exclusivamente para auditoría y particionado.

La descarga completa del repositorio se efectuó en el trabajo de preparación original. Para reutilizar este cuaderno no es necesario repetir una transferencia de varios gigabytes: basta con proporcionar la carpeta original o el ZIP preparado.

In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import io
import json
import random
import zipfile

import numpy as np
import pandas as pd

SEED = 42
DATASET_ID = "PoolC/1-fold-clone-detection-600k-5fold"
RAW_DIR_OPTIONS = [
    Path("/data/notebook_files/poolc_full/data"),
    Path("poolc_full/data"),
]
ZIP_OPTIONS = [
    Path("/data/notebook_files/poolc_prepared.zip"),
    Path("poolc_prepared.zip"),
    Path("/mnt/data/poolc_prepared.zip"),
]
RAW_DIR = next((p for p in RAW_DIR_OPTIONS if p.is_dir()), None)
ZIP_PATH = next((p for p in ZIP_OPTIONS if p.is_file()), None)
RAW_AVAILABLE = bool(RAW_DIR and list(RAW_DIR.glob("train-*.parquet")))
assert RAW_AVAILABLE or ZIP_PATH is not None, "Se requiere la carpeta Parquet original o poolc_prepared.zip."
print("Procedimiento:", "reconstrucción desde Parquet" if RAW_AVAILABLE else "auditoría del ZIP preparado")
print("Semilla aleatoria:", SEED)

Procedimiento: auditoría del ZIP preparado
Semilla aleatoria: 42


## 2. Inspección del esquema

La comprobación del esquema antecede al muestreo para evitar asumir nombres de columnas inexistentes. Si se trabaja con el ZIP, se revisa directamente la estructura exportada y se distingue entre las siete variables de origen y las cuatro columnas de control añadidas durante la preparación.

In [2]:
EXPECTED = {"code1", "code2", "similar", "pair_id", "question_pair_id", "code1_group", "code2_group"}
if RAW_AVAILABLE:
    import pyarrow.parquet as pq
    train_files = sorted(RAW_DIR.glob("train-*.parquet"))
    first_file = pq.ParquetFile(train_files[0])
    print("Archivos de entrenamiento originales:", len(train_files))
    print(first_file.schema_arrow)
    assert EXPECTED.issubset(first_file.schema_arrow.names)
else:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        members = archive.namelist()
        print("Archivos disponibles:", members)
        first_table = pd.read_csv(io.BytesIO(archive.read("poolc_prepared/train.csv")), nrows=3)
    print("Columnas del conjunto preparado:", list(first_table.columns))
    assert EXPECTED.issubset(first_table.columns)

Archivos disponibles: ['poolc_prepared/', 'poolc_prepared/train.csv', 'poolc_prepared/summary.json', 'poolc_prepared/test.csv', 'poolc_prepared/validation.csv']
Columnas del conjunto preparado: ['code1', 'code2', 'similar', 'pair_id', 'question_pair_id', 'code1_group', 'code2_group', 'partition', 'hash1', 'hash2', 'pair_hash']


## 3. Delimitación de grupos y particiones

Los grupos de procedencia permiten impedir que el mismo grupo figure en distintas particiones. En el procedimiento de extracción se recopilan únicamente las columnas de grupos para reducir memoria, se barajan con la semilla 42 y se asignan en proporción aproximada 70 %, 15 % y 15 %.

Esta decisión es más conservadora que separar filas al azar. No obstante, una separación por grupos no garantiza independencia semántica absoluta entre problemas de programación distintos.

In [3]:
if RAW_AVAILABLE:
    import pyarrow.parquet as pq
    rng = np.random.default_rng(SEED)
    all_groups = set()
    for file in train_files:
        parquet = pq.ParquetFile(file)
        for batch in parquet.iter_batches(batch_size=50000, columns=["code1_group", "code2_group"]):
            for column in batch.columns:
                all_groups.update(value for value in column.to_pylist() if value is not None)
    shuffled_groups = sorted(all_groups)
    rng.shuffle(shuffled_groups)
    total_groups = len(shuffled_groups)
    n_training = int(0.70 * total_groups)
    n_validation = int(0.15 * total_groups)
    group_sets = {
        "train": set(shuffled_groups[:n_training]),
        "validation": set(shuffled_groups[n_training:n_training + n_validation]),
        "test": set(shuffled_groups[n_training + n_validation:]),
    }
    group_to_split = {identifier: name for name, ids in group_sets.items() for identifier in ids}
    print({name: len(ids) for name, ids in group_sets.items()})
else:
    print("Los grupos ya fueron separados en el proceso original; la auditoría comprueba su independencia.")

Los grupos ya fueron separados en el proceso original; la auditoría comprueba su independencia.


## 4. Muestreo balanceado por reservorio

Se fijó una meta de 3 000 pares: 2 100 para entrenamiento y 450 para cada partición de validación y prueba. En cada partición, las clases 0 y 1 tienen igual frecuencia. El muestreo por reservorio permite seleccionar elementos sin retener todos los millones de pares simultáneamente en memoria.

Solo se acepta un par cuando **ambos fragmentos pertenecen a grupos asignados a la misma partición**. Los pares que cruzan particiones se excluyen. El balance de clases facilita comparar modelos, pero no reproduce necesariamente la prevalencia de similitud en entornos reales.

In [4]:
TARGETS = {
    "train": {0: 1050, 1: 1050},
    "validation": {0: 225, 1: 225},
    "test": {0: 225, 1: 225},
}
if RAW_AVAILABLE:
    random.seed(SEED)
    reservoirs = {(split, label): [] for split in TARGETS for label in (0, 1)}
    counts = Counter()
    rejected = Counter()
    source_columns = ["code1", "code2", "similar", "pair_id", "question_pair_id", "code1_group", "code2_group"]
    for file in train_files:
        parquet = pq.ParquetFile(file)
        for batch in parquet.iter_batches(batch_size=2000, columns=source_columns):
            batch_dict = batch.to_pydict()
            for row in zip(*(batch_dict[column] for column in source_columns)):
                record = dict(zip(source_columns, row))
                label = record["similar"]
                g1, g2 = record["code1_group"], record["code2_group"]
                if (label not in (0, 1) or not isinstance(record["code1"], str)
                        or not isinstance(record["code2"], str) or not record["code1"].strip()
                        or not record["code2"].strip() or g1 is None or g2 is None):
                    rejected["registros_invalidos"] += 1
                    continue
                p1, p2 = group_to_split.get(g1), group_to_split.get(g2)
                if p1 is None or p1 != p2:
                    continue
                key = (p1, int(label))
                counts[key] += 1
                bucket = reservoirs[key]
                target = TARGETS[p1][int(label)]
                if len(bucket) < target:
                    bucket.append(record)
                else:
                    index = random.randrange(counts[key])
                    if index < target:
                        bucket[index] = record
    parts = []
    for (split, label), records in reservoirs.items():
        assert len(records) == TARGETS[split][label], (split, label, len(records))
        fragment = pd.DataFrame(records)
        fragment["partition"] = split
        parts.append(fragment)
    prepared = pd.concat(parts, ignore_index=True)
    print("Pares elegibles:", dict(counts))
    print("Registros rechazados:", dict(rejected))
else:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        parts = [pd.read_csv(io.BytesIO(archive.read(f"poolc_prepared/{name}.csv")))
                 for name in ("train", "validation", "test")]
    prepared = pd.concat(parts, ignore_index=True)
    print("Pares cargados del ZIP preparado:", len(prepared))

Pares cargados del ZIP preparado: 3000


## 5. Control de calidad e identificación reproducible

Las huellas SHA 256 representan el contenido exacto de cada fragmento. La huella de par se forma ordenando las dos huellas de código, de modo que el intercambio de orden no produzca un par artificialmente nuevo. Se verifican vacíos, etiquetas incompatibles, duplicados y conflictos de etiquetas.

Los identificadores `pair_id` y `question_pair_id` pueden faltar de manera sistemática. Se reporta esa ausencia sin inventar valores y sin utilizar esos campos para el aprendizaje automático.

In [5]:
def digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

prepared["hash1"] = prepared["code1"].map(digest)
prepared["hash2"] = prepared["code2"].map(digest)
prepared["pair_hash"] = ["|".join(sorted((left, right)))
                         for left, right in zip(prepared["hash1"], prepared["hash2"])]

assert len(prepared) == 3000, "La muestra debe contener 3 000 pares."
assert set(prepared["similar"].dropna().astype(int).unique()) == {0, 1}
assert prepared[["code1", "code2", "similar", "code1_group", "code2_group"]].notna().all().all()
assert prepared["code1"].str.strip().str.len().gt(0).all()
assert prepared["code2"].str.strip().str.len().gt(0).all()
assert not prepared["pair_hash"].duplicated().any(), "Se detectaron pares duplicados."
assert not prepared.groupby("pair_hash")["similar"].nunique().gt(1).any()

print("Distribución por partición y clase:")
display(pd.crosstab(prepared["partition"], prepared["similar"], margins=True))
print("Datos faltantes por columna:")
display(prepared.isna().sum().rename("nulos").to_frame())
print("Pares duplicados: 0; etiquetas contradictorias: 0")

Distribución por partición y clase:


similar,0,1,All
partition,,,
test,225,225,450
train,1050,1050,2100
validation,225,225,450
All,1500,1500,3000


Datos faltantes por columna:


,nulos
code1,0
code2,0
similar,0
pair_id,1500
question_pair_id,1500
code1_group,0
code2_group,0
partition,0
hash1,0
hash2,0


Pares duplicados: 0; etiquetas contradictorias: 0


## 6. Verificación de aislamiento entre particiones

La auditoría final compara tanto los identificadores de grupo como las huellas exactas de código. El resultado esperado para cada combinación de particiones es cero intersecciones. Esta prueba reduce la posibilidad de que el modelo memorice fragmentos presentes en más de un conjunto.

La comprobación no detecta equivalencia semántica entre fragmentos distintos ni sustituye una validación externa.

In [6]:
splits = {name: prepared[prepared["partition"] == name].copy()
          for name in ("train", "validation", "test")}
for name, frame in splits.items():
    target = TARGETS[name]
    assert frame["similar"].value_counts().to_dict() == target

for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    a, b = splits[left], splits[right]
    groups_a = set(a.code1_group) | set(a.code2_group)
    groups_b = set(b.code1_group) | set(b.code2_group)
    codes_a = set(a.hash1) | set(a.hash2)
    codes_b = set(b.hash1) | set(b.hash2)
    group_intersection = len(groups_a & groups_b)
    code_intersection = len(codes_a & codes_b)
    print(f"{left} frente a {right}: {group_intersection} grupos compartidos; {code_intersection} códigos idénticos")
    assert group_intersection == 0 and code_intersection == 0
print("Auditoría de particiones completada.")

train frente a validation: 0 grupos compartidos; 0 códigos idénticos
train frente a test: 0 grupos compartidos; 0 códigos idénticos
validation frente a test: 0 grupos compartidos; 0 códigos idénticos
Auditoría de particiones completada.


## 7. Exportación e insumos para el modelo

En el entorno de Datalore los datos se almacenan como tres CSV, con un manifiesto de distribución y un ZIP. El segundo notebook lee directamente este archivo y conserva las mismas particiones. La meta no es maximizar el tamaño del conjunto, sino obtener un experimento verificable dentro de recursos de cómputo modestos.

In [7]:
import os
OUTPUT_DIR = Path("/data/notebook_files/poolc_prepared") if Path("/data/notebook_files").is_dir() else Path("poolc_prepared")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {}
for name, frame in splits.items():
    path = OUTPUT_DIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    manifest[name] = {
        "total_pairs": len(frame),
        "class_distribution": {str(key): int(value) for key, value in frame.similar.value_counts().items()}
    }
with (OUTPUT_DIR / "summary.json").open("w", encoding="utf-8") as output:
    json.dump(manifest, output, indent=2, ensure_ascii=False)
ZIP_OUTPUT = OUTPUT_DIR.parent / "poolc_prepared_verificado.zip"
with zipfile.ZipFile(ZIP_OUTPUT, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in ("train.csv", "validation.csv", "test.csv", "summary.json"):
        archive.write(OUTPUT_DIR / filename, arcname=f"poolc_prepared/{filename}")
print("Archivos listos en:", OUTPUT_DIR)
print("ZIP verificado:", ZIP_OUTPUT)

Archivos listos en: poolc_prepared
ZIP verificado: poolc_prepared_verificado.zip


## 8. Interpretación y límites de la preparación

El experimento utiliza una muestra balanceada extraída de los archivos originales `train` de PoolC, no la totalidad del repositorio. El tamaño de la muestra y su balance fueron decisiones instrumentales para entrenar en CPU y comparar modelos bajo condiciones reproducibles. La validación se reservará para elegir el umbral, y la prueba se utilizará una sola vez para estimar el desempeño final.

El repositorio público no presenta una ficha detallada con la justificación completa de sus etiquetas ni con una licencia explícita fácilmente verificable. Antes de utilizar los datos en un producto comercial debe confirmarse la titularidad y el permiso de uso. Para referencias se consultaron la página del dataset, documentación de Python y scikit-learn.

**Resultado de esta etapa:** un conjunto de 3 000 pares distribuidos en 2 100 para entrenamiento, 450 para validación y 450 para prueba, con verificación de duplicación y separación de grupos.